## Setup

First, let's install the required packages and set up the API keys

In [ ]:
%%capture --no-stderr
%pip install -U langchain-openai langgraph langgraph-checkpoint-redis

In [ ]:
import getpass
import os

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

## Graph implementation

Let's implement the graph that will leverage Redis for storing state transitions.

In [ ]:
from IPython.display import Image, display
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.checkpoint.redis import RedisSaver
from langgraph.prebuilt import create_react_agent

model = ChatOpenAI(model="gpt-4o", temperature=0)

@tool
def get_weather(location: str) -> str:
    """Use this to get weather information."""
    if any([city in location.lower() for city in ["nyc", "new york city"]]):
        return "It might be cloudy in nyc"
    elif any([city in location.lower() for city in ["sf", "san francisco"]]):
        return "It's always sunny in sf"
    else:
        return f"I am not sure what the weather is in {location}"

tools = [get_weather]

REDIS_URI = "redis://localhost:6379"
checkpointer = None
with RedisSaver.from_conn_string(REDIS_URI) as _checkpointer:
    _checkpointer.setup()
    checkpointer = _checkpointer

graph = create_react_agent(model, tools=tools, checkpointer=checkpointer)
display(Image(graph.get_graph().draw_mermaid_png()))

## Usage

Let's interact with the graph to show it can remember

In [ ]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [ ]:
config = {"configurable": {"thread_id": "1"}}
inputs = {"messages": [("user", "What's the weather in NYC?")]}

print_stream(graph.stream(inputs, config=config, stream_mode="values"))

Notice that when we pass the same thread ID, the chat history is preserved.

In [ ]:
inputs = {"messages": [("user", "What's it known for?")]}
print_stream(graph.stream(inputs, config=config, stream_mode="values"))